# Resume Scorer — Interactive Test

Use this notebook to test the Resume Scorer agent with different job URLs and resumes.

Every run is **automatically traced in LangSmith** — check your project dashboard to inspect
node-level inputs, outputs, and latency.

---
**Workflow:** `scrape_job` → `run_scorer`

| Node | What it does |
|---|---|
| `scrape_job` | Fetches and cleans the job description from the URL |
| `run_scorer` | Calls Gemini 2.5 Flash and returns a structured JSON assessment |

## Setup

Loads environment variables (`.env`) and imports the graph.
Run this cell once before anything else.

In [3]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add src/ to path so the agent package is importable when running outside Docker
sys.path.insert(0, str(Path("../src").resolve()))

# Load API keys from Agent/.env
load_dotenv("../.env")

from agent.graph import graph

print("Graph loaded. READY TO RUN BABY.")

Graph loaded. READY TO RUN BABY.


---
## 1. Define Your Inputs

Replace `JOB_URL` with a real posting and paste the resume text below.

In [4]:
JOB_URL = "https://www.linkedin.com/jobs/collections/recommended/?currentJobId=4386489615&discover=recommended&discoveryOrigin=JOBS_HOME_JYMBII"

RESUME_TEXT = """
Caiua Utida  |  utida@gmail.com.com  |  linkedin.com/in/utida

EXPERIENCE
AI Engineer — Padrinhos Magicos Corp  (2021 – Present)
  • Built LLMs workflows with FastAPI and LangChain serving 50 000 users/day.
  • Led integration from scratch to end-to-end using Docker.
  • Reduced deployment time by 40 % with a GitHub Actions CI/CD pipeline.

Data Scientist Junior — Alpha Inc  (2019 – 2022)
  • Developed internal dashboards using Python.
  • Developed Deep Learning models for customer segmentation, improving targeting by 25 %.

SKILLS
Python, FastAPI, LangChain, Deep Learning, Docker, GitHub Actions, Data Visualization

EDUCATION
B.Sc. Computer Science — State University  (2019)
"""

---
## 2. Run the Agent

This will scrape the job URL and call Gemini 2.5 Flash.
The full trace will appear in your LangSmith dashboard under the `resume-scorer` project.

In [5]:
result = await graph.ainvoke({
    "job_url": JOB_URL,
    "resume_text": RESUME_TEXT,
})

score = result["score_result"]
print(json.dumps(score, indent=2))

{
  "match_score": 40,
  "matched_skills": [
    "Python",
    "Deep Learning",
    "Generative AI",
    "Data Visualization",
    "Docker",
    "GitHub Actions"
  ],
  "missing_skills": [
    "SQL",
    "Numpy",
    "Pandas",
    "Scikit-learn",
    "Basic Statistics",
    "Git",
    "Jupyter Notebooks",
    "Kanban",
    "Big Data (specific tools/experience)"
  ],
  "experience_gap": "The candidate's 4.5 years of professional experience and completed B.Sc. from 2019 significantly exceed the JD's requirement for a student currently between the 3rd and 6th academic period.",
  "top_improvements": [
    {
      "action": "Clarify current student status or provide a compelling reason for applying for an internship despite extensive professional experience",
      "reason": "The job description explicitly requires candidates to be current students within a specific academic range, which is a fundamental eligibility criterion.",
      "priority": "high"
    },
    {
      "action": "Explic

---
## 3. Quick Summary

In [6]:
print(f"Match score : {score['match_score']}/100")
print(f"Summary     : {score['summary']}")
print()
print("Matched skills  :", ", ".join(score["matched_skills"]))
print("Missing skills  :", ", ".join(score["missing_skills"]))
print("Experience gap  :", score["experience_gap"])
print()
print("Top improvements:")
for i, item in enumerate(score["top_improvements"], 1):
    print(f"  {i}. [{item['priority'].upper()}] {item['action']}")
    print(f"     → {item['reason']}")

Match score : 40/100
Summary     : The candidate demonstrates strong technical proficiency in Python, Deep Learning, Generative AI, and data visualization, aligning well with many of the core technical responsibilities of the role. However, the candidate's completed B.Sc. and several years of professional experience significantly exceed the internship's strict requirement for a student currently between the 3rd and 6th academic period, indicating a fundamental mismatch in seniority and eligibility.

Matched skills  : Python, Deep Learning, Generative AI, Data Visualization, Docker, GitHub Actions
Missing skills  : SQL, Numpy, Pandas, Scikit-learn, Basic Statistics, Git, Jupyter Notebooks, Kanban, Big Data (specific tools/experience)
Experience gap  : The candidate's 4.5 years of professional experience and completed B.Sc. from 2019 significantly exceed the JD's requirement for a student currently between the 3rd and 6th academic period.

Top improvements:
  1. [HIGH] Clarify current st